# Analytic Knot Fields and Deformations

Use this notebook when your starting object is a **knot/link type, Artin braid, or complex analytic field** rather than a pre-existing mesh or graph.

The generic workflow is

\[
\text{knot / braid}\to f(u,v,\bar v)\to F(x,y,z)
\to \{|F|\le\epsilon\}\to \text{spatial graph}
\to \text{projection}\to \Upsilon.
\]

This is an additive input route: once `to_spatial_graph(...)` returns an embedded `MultiGraph`, the library uses the same optimized extraction, projection, and Yamada stack as the other User Guide workflows.

Generic \(S^3/\mathbb R^3\) knot fields are **not automatically periodic Bloch Hamiltonians**. The final section shows the separate Bloch-deformation API for physical nodal models.


In [ ]:
import numpy as np
import sympy as sp

from knotted_graph.inputs import (
    KnotFunction,
    KnotFunctionPath,
    available_knot_names,
)
from knotted_graph.applications.knot_deformation import KnotDeformationScan
from knotted_graph.projection import compute_yamada_polynomial

Y = sp.Symbol("Y")
print("built-in names:", available_knot_names())


## 1. Construct knots and links

Preferred named constructors use exact/reference fields when available. `from_braid(...)` is the general route for closures of arbitrary Artin braid words. Positive `i` means \(\sigma_i\); negative `-i` means \(\sigma_i^{-1}\).


In [ ]:
trefoil = KnotFunction.from_name("3_1")
figure8 = KnotFunction.from_name("4_1")
torus_25 = KnotFunction.torus(2, 5)

figure8_braid = KnotFunction.from_braid(
    [1, -2, 1, -2],
    strands=3,
    fourier_modes=(4, 8, 12, 16),
    validation_samples=256,
)

report = figure8_braid.construction_report
print("construction:", figure8.metadata["construction"])
print("braid validation passed:", report.passed)
print("root error / separation:", report.error_fraction)
print(report.interpretation)


The braid report validates the **finite Fourier realization on sampled braid parameters**. It is not a formal certificate for the theorem's unspecified sufficiently-small \(S^3\) scaling threshold. For publication-grade use, also verify the sampled 3-D tubular topology and resolution convergence.

## 2. Build a tubular handlebody and inspect it

For a complex field \(F\),

\[
H_\epsilon=\{|F|\le\epsilon\},\qquad \partial H_\epsilon=\{|F|=\epsilon\}.
\]

For sufficiently small regular \(\epsilon\), this is a tubular neighborhood of the link. Numerical grids must still be checked for connected components, closed boundaries, genus, and box-boundary contact.

A publication-grade figure-eight check is:

```python
diagnostic = figure8.diagnose_level(
    0.55,
    span=((-4, 4),) * 3,
    dimension=160,
)

report = figure8.tubular_convergence(
    0.55,
    span=((-4, 4),) * 3,
    dimensions=(128, 160),
)
assert report.converged
```

The high-resolution convergence call is shown rather than executed automatically because it is intentionally expensive.


In [ ]:
sample = figure8.sample(span=((-4, 4),) * 3, dimension=24)
print("sample shape:", sample.values.shape)
print("field min/max |F|:", float(sample.abs_values.min()), float(sample.abs_values.max()))


## 3. Convert the handlebody to the optimized spatial-graph pipeline

After validating the level radius, convert it with:

```python
graph = figure8.to_spatial_graph(
    radius=0.55,
    span=((-4, 4),) * 3,
    dimension=160,
)
```

Internally this calls the current canonical `skeletonize_volume(...)` and `skeleton_image_to_graph(...)`. It does not carry a separate skeletonizer.

The returned node `pos` values and edge `pts` polylines are mapped back from voxel indices into the physical coordinates of the sampled field. From this point onward, use exactly the standard Core Workflows API:

```python
yamada = compute_yamada_polynomial(graph, Y)
```


## 4. Deform one analytic knot field into another

`KnotFunctionPath` RMS-normalizes the endpoint fields and aligns their global complex phase before linear interpolation,

\[
F_\lambda=(1-\lambda)\widehat F_0+\lambda\widehat F_1.
\]

This removes arbitrary overall scale/phase choices, but **does not make the homotopy canonical**: intermediate topology belongs to the chosen field representatives and path, not merely the endpoint knot types.


In [ ]:
path = KnotFunctionPath(
    KnotFunction.from_name("3_1"),
    KnotFunction.from_name("4_1"),
    sample_count=256,
)

mid = path.at(0.5)
probe = mid(0.2, -0.1, 0.3)
print("mid-path field at probe point:", complex(np.asarray(probe).item()))
print("aligned overlap:", path.gauge.overlap_after_alignment)


A two-parameter topology scan is then:

```python
scan = KnotDeformationScan(
    path,
    lambdas=np.linspace(0, 1, 31),
    radii=np.linspace(0.1, 0.5, 21),
    span=((-4, 4),) * 3,
    dimension=128,
    invariant="yamada",
)
result = scan.run()
result.plot_phase_diagram()
```

Each point follows

\[
F_\lambda\to H_{\lambda,\epsilon}\to G_{\lambda,\epsilon}\to \Upsilon(G_{\lambda,\epsilon}).
\]


## 5. Physical/Bloch deformations are a separate layer

For periodic nodal models, deform the Bloch vectors themselves rather than assuming a generic \(S^3/\mathbb R^3\) knot field is a Brillouin-zone Hamiltonian.


In [ ]:
from knotted_graph.applications.nodal import (
    NodalBlochPath,
    hopf_link_bloch_vector,
    unknot_bloch_vector,
)

bloch_path = NodalBlochPath(
    unknot_bloch_vector,
    hopf_link_bloch_vector,
)

vec = bloch_path.at(gamma=0.2, lam=0.4)
vec_xyz = bloch_path.at_components(
    gamma=0.2,
    weights=(0.2, 0.5, 0.8),
)

print("uniform lambda components:", vec)
print("component-wise lambda components:", vec_xyz)


For the full \((\lambda,\gamma)\) Yamada phase scan:

```python
from knotted_graph.applications.nodal import NodalPhaseScan

scan = NodalPhaseScan(
    bloch_path,
    lambdas=np.linspace(0, 1, 41),
    gammas=np.linspace(0.1, 1.0, 30),
    dimension=96,
)
result = scan.run()
```

This is the reusable library form of the earlier phase-space notebook calculation, while the generic arbitrary-knot field API remains mathematically separate from the periodic Bloch realization.
